In [1]:
import io
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Image

# ── change this path to load a different trial ────────────────────────────
FILE_PATH = "/home/metamobility3/Jinwoo/os_kinetics/cascade_0502_gain_0p3.npz"
# FILE_PATH = "/home/metamobility3/Jinwoo/os_kinetics/cascade_0502_gain_0p5.npz"
# ──────────────────────────────────────────────────────────────────────────

In [2]:
d = np.load(FILE_PATH, allow_pickle=True)
print(f"Loaded : {os.path.basename(FILE_PATH)}")
print(f"Keys   : {list(d.keys())}")

time   = d["time"]
dt     = float(np.median(np.diff(time)))
fs     = 1.0 / dt
dur    = time[-1] - time[0]
n      = len(time)

# motor encoders (rad)
hip_pos_R = d["hip_pos_R"]
hip_pos_L = d["hip_pos_L"]

# motor velocities (rad/s)
hip_vel_R = d["hip_vel_R"]
hip_vel_L = d["hip_vel_L"]

# IMU gyros — Gyr_Y (index 4, deg/s) used for hip flex/ext velocity
gyro_P = d["imu_P"][:, 4]   # pelvis
gyro_R = d["imu_R"][:, 4]   # right thigh
gyro_L = d["imu_L"][:, 4]   # left thigh

# model I/O  (right-side reference channel logged by main_hip.py)
model_in_angle = d["model_in_angle_raw"]   # rad
model_in_vel   = d["model_in_vel_raw"]     # rad/s
model_out_R    = d["model_out_R"]          # Nm/kg
model_out_L    = d["model_out_L"]          # Nm/kg

# commands and applied torques (Nm)
cmd_R     = d["cmd_R"]
cmd_L     = d["cmd_L"]
applied_R = d["applied_R"]
applied_L = d["applied_L"]

# motion gate
assist_gate  = d["assist_gate"]
motion_score = d["motion_score"]
state        = d["state"]
gpio         = d["GPIO"]

print(f"Duration : {dur:.1f} s   fs ≈ {fs:.1f} Hz   samples : {n}")
print(f"cmd_R    : [{cmd_R.min():.2f}, {cmd_R.max():.2f}] Nm")
print(f"cmd_L    : [{cmd_L.min():.2f}, {cmd_L.max():.2f}] Nm")

Loaded : cascade_0502_gain_0p3.npz
Keys   : ['time', 'hip_pos_L', 'hip_pos_R', 'hip_vel_L', 'hip_vel_R', 'cmd_L', 'cmd_R', 'model_out_L', 'model_out_R', 'applied_L', 'applied_R', 'imu_P', 'imu_L', 'imu_R', 'GPIO', 'model_in_angle_raw', 'model_in_vel_raw', 'model_out_nmpkg', 'moment_raw', 'assist_gate', 'motion_score', 'state']
Duration : 36.1 s   fs ≈ 100.0 Hz   samples : 3609
cmd_R    : [-4.50, 2.34] Nm
cmd_L    : [-1.68, 1.20] Nm


In [3]:
PALETTE = {
    "R"     : "#2196F3",
    "L"     : "#FF9800",
    "model" : "#9C27B0",
    "gate"  : "#4CAF50",
    "score" : "#795548",
    "gpio"  : "#607D8B",
    "pelvis": "#E91E63",
}

out = widgets.Output()

slider = widgets.FloatRangeSlider(
    value=[time[0], time[-1]],
    min=time[0], max=time[-1], step=0.1,
    description="Time (s):",
    layout=widgets.Layout(width="760px"),
    style={"description_width": "initial"},
    continuous_update=False,
    readout_format=".1f",
)


def _draw(t0, t1):
    m = (time >= t0) & (time <= t1)
    t = time[m]

    fig, axs = plt.subplots(6, 1, figsize=(16, 18), sharex=True)

    # 1 — Hip encoder angles
    axs[0].plot(t, np.rad2deg(hip_pos_R[m]), color=PALETTE["R"], lw=1.5, label="Right")
    axs[0].plot(t, np.rad2deg(hip_pos_L[m]), color=PALETTE["L"], lw=1.5, label="Left")
    axs[0].axhline(0, color="gray", lw=0.7, ls="--")
    axs[0].set_ylabel("Hip angle (deg)", fontsize=12)
    axs[0].legend(fontsize=10, loc="upper right")

    # 2 — Model inputs (right-side reference)
    ax2b = axs[1].twinx()
    axs[1].plot(t, np.rad2deg(model_in_angle[m]), color=PALETTE["R"],     lw=1.5, label="Hip angle — model in (deg)")
    ax2b.plot(   t, np.rad2deg(model_in_vel[m]),  color=PALETTE["model"], lw=1.2, ls="--", label="Hip vel — model in (deg/s)")
    axs[1].axhline(0, color="gray", lw=0.7, ls="--")
    axs[1].set_ylabel("Angle (deg)", fontsize=12)
    ax2b.set_ylabel("Vel (deg/s)",  fontsize=12, color=PALETTE["model"])
    lines1, labels1 = axs[1].get_legend_handles_labels()
    lines2, labels2 = ax2b.get_legend_handles_labels()
    axs[1].legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc="upper right")

    # 3 — Model outputs (Nm/kg)
    axs[2].plot(t, model_out_R[m], color=PALETTE["R"], lw=1.5, label="model_out R (Nm/kg)")
    axs[2].plot(t, model_out_L[m], color=PALETTE["L"], lw=1.5, label="model_out L (Nm/kg)")
    axs[2].axhline(0, color="gray", lw=0.7, ls="--")
    axs[2].set_ylabel("Model out (Nm/kg)", fontsize=12)
    axs[2].legend(fontsize=10, loc="upper right")

    # 4 — Motor commands (Nm)
    axs[3].plot(t, cmd_R[m],     color=PALETTE["R"], lw=1.5, label="cmd R (Nm)")
    axs[3].plot(t, cmd_L[m],     color=PALETTE["L"], lw=1.5, label="cmd L (Nm)")
    axs[3].axhline(0, color="gray", lw=0.7, ls="--")
    axs[3].set_ylabel("Motor cmd (Nm)", fontsize=12)
    axs[3].legend(fontsize=10, loc="upper right")

    # 5 — IMU Gyr_Y (deg/s) for all three IMUs
    axs[4].plot(t, gyro_R[m], color=PALETTE["R"],     lw=1.2, label="Thigh R Gyr_Y")
    axs[4].plot(t, gyro_L[m], color=PALETTE["L"],     lw=1.2, label="Thigh L Gyr_Y")
    axs[4].plot(t, gyro_P[m], color=PALETTE["pelvis"], lw=1.2, label="Pelvis Gyr_Y")
    axs[4].axhline(0, color="gray", lw=0.7, ls="--")
    axs[4].set_ylabel("Gyr_Y (deg/s)", fontsize=12)
    axs[4].legend(fontsize=10, loc="upper right")

    # 6 — Assist gate, motion score, state, GPIO
    axs[5].plot(t, assist_gate[m],  color=PALETTE["gate"],  lw=1.5, label="Assist gate")
    axs[5].plot(t, motion_score[m], color=PALETTE["score"], lw=1.2, ls="--", label="Motion score")
    axs[5].plot(t, state[m] / 2,    color="#FF5722",        lw=1.0, ls=":",  label="State /2")
    axs[5].plot(t, gpio[m],         color=PALETTE["gpio"],  lw=1.5, label="GPIO")
    axs[5].set_ylabel("Gate / score", fontsize=12)
    axs[5].set_xlabel("Time (s)", fontsize=12)
    axs[5].legend(fontsize=10, loc="upper right")

    for ax in axs:
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(labelsize=10)

    fig.suptitle(os.path.basename(FILE_PATH), fontsize=13)
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    out.clear_output(wait=True)
    with out:
        display(Image(data=buf.read()))


def _on_slider(change):
    _draw(*slider.value)

slider.observe(_on_slider, names="value")
display(slider, out)
_draw(time[0], time[-1])

FloatRangeSlider(value=(0.010097946999849228, 36.090059809000195), continuous_update=False, description='Time …

Output()

In [4]:
# ── summary statistics for the current slider window ─────────────────────
t0_s, t1_s = slider.value
m = (time >= t0_s) & (time <= t1_s)

signals = {
    "Hip angle R (deg)"      : np.rad2deg(hip_pos_R[m]),
    "Hip angle L (deg)"      : np.rad2deg(hip_pos_L[m]),
    "Model in angle (deg)"   : np.rad2deg(model_in_angle[m]),
    "Model in vel (deg/s)"   : np.rad2deg(model_in_vel[m]),
    "Model out R (Nm/kg)"    : model_out_R[m],
    "Model out L (Nm/kg)"    : model_out_L[m],
    "Cmd R (Nm)"             : cmd_R[m],
    "Cmd L (Nm)"             : cmd_L[m],
    "Assist gate"            : assist_gate[m],
    "Motion score"           : motion_score[m],
}

print(f"Window: {t0_s:.1f} – {t1_s:.1f} s  ({m.sum()} samples)")
print(f"{'Signal':<26} {'min':>8} {'max':>8} {'mean':>8} {'std':>8}")
print("-" * 62)
for name, sig in signals.items():
    print(f"{name:<26} {sig.min():8.3f} {sig.max():8.3f} {sig.mean():8.3f} {sig.std():8.3f}")

Window: 0.0 – 36.1 s  (3609 samples)
Signal                          min      max     mean      std
--------------------------------------------------------------
Hip angle R (deg)           -26.200   39.300    6.551   17.673
Hip angle L (deg)           -30.700   21.200   -4.719   13.155
Model in angle (deg)        -26.200   39.300    6.551   17.673
Model in vel (deg/s)       -224.809  306.229    0.992   99.558
Model out R (Nm/kg)          -0.660    0.311   -0.069    0.241
Model out L (Nm/kg)          -0.213    0.160   -0.020    0.071
Cmd R (Nm)                   -4.500    2.340   -0.542    1.842
Cmd L (Nm)                   -1.683    1.201   -0.155    0.546
Assist gate                   0.000    0.000    0.000    0.000
Motion score                  0.000    0.000    0.000    0.000
